## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# ⚠️ Exception Handling with Middleware

## Industry Use Case: Market Data Service Recovery

This notebook demonstrates exception handling for unreliable external services.

| Feature | FSI Application |
|---------|-----------------|
| **Error Catching** | Handle market data API timeouts |
| **Graceful Degradation** | Provide fallback responses |
| **Error Logging** | Track service failures |

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

PROJECT_ENDPOINT = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")

print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print("✅ Environment loaded")


In [ ]:
from collections.abc import Awaitable, Callable

from agent_framework import Agent, FunctionInvocationContext
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print("✅ All imports loaded")

## Define Unstable Market Data Service

In [ ]:
def get_stock_price(symbol: str) -> str:
    """Get stock price - simulates an unreliable market data service."""
    # Simulate a service timeout
    raise TimeoutError("Market data service request timed out")

print("✅ Tool defined: get_stock_price (will always timeout)")

## Define Exception Handling Middleware

In [ ]:
async def exception_handling_middleware(
    context: FunctionInvocationContext,
    next: Callable[[], Awaitable[None]],
) -> None:
    """Middleware that catches exceptions and provides graceful fallbacks."""
    function_name = context.function.name

    try:
        print(f"[ExceptionHandler] Executing: {function_name}")
        await next()
        print(f"[ExceptionHandler] {function_name} completed successfully")
    except TimeoutError as e:
        print(f"[ExceptionHandler] Caught TimeoutError: {e}")
        # Override the tool result with a user-friendly fallback message
        context.result = (
            "Market data service is temporarily unavailable. "
            "Please try again later or contact support."
        )

print("✅ Middleware defined: exception_handling_middleware")


## Run Example

In [ ]:
async def main():
    print("=== Exception Handling Middleware Example ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        try:
            agent = Agent(
                client=client,
                name="MarketDataAgent",
                instructions="You are a market data assistant. Use get_stock_price to fetch prices.",
                tools=get_stock_price,
                middleware=[exception_handling_middleware],
            )

            query = "What is the current price of MSFT stock?"
            print(f"User: {query}")
            result = await agent.run(query)
            print(f"Agent: {result.text if result.text else 'No response'}")
        finally:
            await client.client.close()
            await client.project_client.close()

await main()


## 📋 Key Takeaways

| Feature | Description |
|---------|-------------|
| **try-except** | Wrap `await next()` to catch errors |
| **context.result** | Override with user-friendly message |
| **Graceful degradation** | Agent responds helpfully even on failure |
